In [1]:
!pip install scikit-learn xgboost

In [2]:
import numpy as np
import pandas as pd

text_emb = np.load("/kaggle/input/notebooks/navishaaagarwaal/02-feature-extraction/text_emb.npy")
img_emb = np.load("/kaggle/input/notebooks/navishaaagarwaal/02-feature-extraction/img_emb.npy")
hash_emb = np.load("/kaggle/input/notebooks/navishaaagarwaal/02-feature-extraction/hash_emb.npy")

hashtag_count = np.load("/kaggle/input/notebooks/navishaaagarwaal/02-feature-extraction/hashtag_count.npy")
hashtag_freq = np.load("/kaggle/input/notebooks/navishaaagarwaal/02-feature-extraction/hashtag_freq.npy")

df = pd.read_csv("/kaggle/input/notebooks/navishaaagarwaal/01-dataset-preparation/labeled.csv")

y = df["label"].values

In [3]:
def normalize(x):
    return x / np.linalg.norm(x, axis=1, keepdims=True)

text_n = normalize(text_emb)
img_n = normalize(img_emb)
hash_n = normalize(hash_emb)

In [4]:
text_img_sim = np.sum(text_n * img_n, axis=1)
text_hash_sim = np.sum(text_n * hash_n, axis=1)
img_hash_sim = np.sum(img_n * hash_n, axis=1)

In [5]:
text_img_diff = np.abs(text_emb - img_emb)
text_hash_diff = np.abs(text_emb - hash_emb)
img_hash_diff = np.abs(img_emb - hash_emb)

In [6]:
text_img_sim = text_img_sim.reshape(-1,1)
text_hash_sim = text_hash_sim.reshape(-1,1)
img_hash_sim = img_hash_sim.reshape(-1,1)

hashtag_count = hashtag_count.reshape(-1,1)
hashtag_freq = hashtag_freq.reshape(-1,1)

In [7]:
X = np.concatenate([
    text_emb,
    img_emb,
    hash_emb,
    text_img_diff,
    text_hash_diff,
    img_hash_diff,
    text_img_sim,
    text_hash_sim,
    img_hash_sim,
    hashtag_count,
    hashtag_freq
], axis=1)

print("Final feature shape:", X.shape)

Final feature shape: (11902, 3077)


In [8]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score
from xgboost import XGBClassifier

In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [10]:
neg = np.sum(y_train == 0)
pos = np.sum(y_train == 1)

pos_weight = neg / pos

In [11]:
model = XGBClassifier(
    n_estimators=400,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=pos_weight,
    eval_metric="logloss",
    tree_method="hist",
    n_jobs=-1
)

model.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.05, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=6, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=400, n_jobs=-1,
              num_parallel_tree=None, ...)

In [12]:
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:,1]

print(classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

              precision    recall  f1-score   support

           0       0.92      0.95      0.93      2083
           1       0.54      0.40      0.46       298

    accuracy                           0.88      2381
   macro avg       0.73      0.67      0.69      2381
weighted avg       0.87      0.88      0.87      2381

ROC-AUC: 0.8299762861386681


In [13]:
np.save("/kaggle/working/X_features.npy", X)
np.save("/kaggle/working/y_labels.npy", y)

In [14]:
import joblib
joblib.dump(model, "/kaggle/working/xgb_model.pkl")

['/kaggle/working/xgb_model.pkl']